Library Import

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

Kaggle import

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dorianlazar/medium-articles-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.33G/1.33G [00:38<00:00, 36.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1


In [3]:
df = pd.read_csv("/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/medium_data.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6508 entries, 0 to 6507
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            6508 non-null   int64 
 1   url           6508 non-null   object
 2   title         6508 non-null   object
 3   subtitle      3479 non-null   object
 4   image         6361 non-null   object
 5   claps         6508 non-null   int64 
 6   responses     6508 non-null   object
 7   reading_time  6508 non-null   int64 
 8   publication   6508 non-null   object
 9   date          6508 non-null   object
dtypes: int64(3), object(7)
memory usage: 508.6+ KB


Only title column is used

In [5]:
df = df['title']

nltk download

In [6]:
!pip  install nltk

In [7]:
# all the title are being joined with \n and objec type is changed into string type
document = "\n".join(df.astype(str))

In [8]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [9]:
from nltk import word_tokenize
tokens = word_tokenize(document.lower())

In [10]:
vocab = {"<unk>" : 0}

In [11]:
# the number are assigned to each tokens in vocab
from collections import Counter
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

In [12]:
# now divide the data
input_sequences = document.split("\n")

In [13]:
def text_to_indices(sentence , vocab):
  numerical_sentence =[]
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab["<unk>"])
  return numerical_sentence

In [14]:
input_numerical_sentences = [];
for sentence in input_sequences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()) , vocab))

In [15]:
# converts all the sentences into numerical values
# input_numerical_sentences

In [16]:
training_sequences = []
for sentence in input_numerical_sentences:
  for i in range(1 , len(sentence)):
    training_sequences.append(sentence[:i+1])

In [17]:
# training_sequences

In [18]:
# using padding so that the size of all are sane
len_list = []
for sequences in training_sequences:
  len_list.append(len(sequences))

In [19]:
padding_training_sequence = []
for sequence in training_sequences:
  padding_training_sequence.append([0] * (max(len_list) - len(sequence)) + sequence)

In [20]:
len(padding_training_sequence[0])

51

In [21]:
padding_training_sequence = torch.tensor(padding_training_sequence , dtype = torch.long)

In [22]:
X = padding_training_sequence[: , :-1]
y = padding_training_sequence[:,-1]

In [23]:
from torch.utils.data import Dataset , DataLoader

class CustomDataset(Dataset):
 def __init__(self,X ,y):
  self.x = X
  self.y = y
 def __len__(self):
  return self.x.shape[0]
 def __getitem__(self, idx):
  return self.x[idx] , self.y[idx]

In [24]:
dataset = CustomDataset(X,y)

In [25]:
dataloader = DataLoader(dataset , batch_size = 32 , shuffle = True)

In [26]:
class LSTMmodel(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size , 100)
    self.lstm = nn.LSTM(100,150,batch_first = True)
    self.fc = nn.Linear(150,vocab_size)
  def forward(self , X):
    embedded = self.embedding(X)
    intermediate_hidden_state , (final_hidden_state , final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [27]:
model = LSTMmodel(len(vocab))

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
device

device(type='cuda')

In [30]:
model.to(device)

LSTMmodel(
  (embedding): Embedding(8347, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=8347, bias=True)
)

In [31]:
epochs = 50
learning_rate = 0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr = learning_rate)

In [34]:
training_loss_history = []
for epoch in range(epochs):
  for X_batch , y_batch in dataloader:
    X_batch , y_batch = X_batch.to(device) , y_batch.to(device)

    y_pred = model(X_batch)

    loss = criterion(y_pred , y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  training_loss_history.append(loss.item())
  print(f"Epoch: {epoch} , Loss: {loss.item()}")

Epoch: 0 , Loss: 4.775213718414307
Epoch: 1 , Loss: 3.8542182445526123
Epoch: 2 , Loss: 5.1918625831604
Epoch: 3 , Loss: 3.5856239795684814
Epoch: 4 , Loss: 3.3248376846313477
Epoch: 5 , Loss: 4.000555038452148
Epoch: 6 , Loss: 4.198526382446289
Epoch: 7 , Loss: 3.556863307952881
Epoch: 8 , Loss: 2.4699296951293945
Epoch: 9 , Loss: 2.9154412746429443
Epoch: 10 , Loss: 1.5004868507385254
Epoch: 11 , Loss: 1.9725319147109985
Epoch: 12 , Loss: 3.090609073638916
Epoch: 13 , Loss: 2.552842140197754
Epoch: 14 , Loss: 2.8265061378479004
Epoch: 15 , Loss: 1.249445915222168
Epoch: 16 , Loss: 2.5770814418792725
Epoch: 17 , Loss: 2.394901990890503
Epoch: 18 , Loss: 1.877500057220459
Epoch: 19 , Loss: 2.931227207183838
Epoch: 20 , Loss: 2.895930767059326
Epoch: 21 , Loss: 3.243443489074707
Epoch: 22 , Loss: 1.3643040657043457
Epoch: 23 , Loss: 3.716876983642578
Epoch: 24 , Loss: 4.476574420928955
Epoch: 25 , Loss: 3.320387363433838
Epoch: 26 , Loss: 5.533320903778076
Epoch: 27 , Loss: 3.9013636112

In [38]:
def generate_next_words(model, vocab, seed_text, num_words):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = model.to(device)
  current_text_tokens = word_tokenize(seed_text.lower())
  generated_words = []

  idx_to_word = {idx: word for word, idx in vocab.items()}

  for _ in range(num_words):
    numerical_text = text_to_indices(current_text_tokens, vocab)
    # Pad the numerical_text to match the maximum sequence length
    padded_text = torch.tensor([0] * (max(len_list) - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
      output = model(padded_text)

    predicted_token_id = torch.argmax(output, dim=1).item()
    predicted_word = idx_to_word.get(predicted_token_id, "<unk>")

    current_text_tokens.append(predicted_word) # Append the predicted word for the next iteration
    generated_words.append(predicted_word)

  return " ".join(generated_words)

In [45]:
seed_text = "Introduction to "
num_words_to_generate = 5

predicted_sequence = generate_next_words(model, vocab, seed_text, num_words_to_generate)
print(f"Seed text: {seed_text}")
print(f"Generated sequence: {seed_text} {predicted_sequence}")

Seed text: Introduction to 
Generated sequence: Introduction to  the attention economy and economic
